# David vs Goliat — CIFAR-10 en Colab GPU

Pipeline completo: **pre-entrenar el diccionario** (sparse coding no supervisado) -> **visualizar los átomos** (deben salir Gabor-like) -> **curva de datos** con todas las condiciones -> **figuras**.

**Antes de empezar:** menú `Entorno de ejecución -> Cambiar tipo de entorno -> GPU (T4)`.

`device.py` detecta CUDA automáticamente (MPS > CUDA > CPU), así que el mismo código del Mac corre en la GPU de Colab sin tocar nada.

In [ ]:
# 1. Confirmar que hay GPU
!nvidia-smi

## 2. Traer el repo (privado)

El repo es privado, así que `git clone` necesita un **token**. En GitHub: `Settings -> Developer settings -> Personal access tokens -> Fine-grained tokens`, dale acceso de solo-lectura al repo `david-vs-goliat`, cópialo y pégalo abajo.

(Alternativa sin token: sube el repo como .zip con el panel de archivos de la izquierda y descomprímelo con `!unzip`.)

In [ ]:
TOKEN = "PEGA_TU_TOKEN_AQUI"  # <-- reemplaza
!git clone https://{TOKEN}@github.com/emazagrandes/david-vs-goliat.git
%cd david-vs-goliat

In [ ]:
# 3. Dependencias (torch/torchvision ya vienen en Colab; falta thop para FLOPs)
!pip install -q thop
# Comprobar que el código ve la GPU
import torch
from src.utils.device import get_device
print('device:', get_device())

## 4. FASE 1 — pre-entrenar el diccionario Φ (sin etiquetas)

Aprende Φ por reconstrucción (`min ½‖x − Φa‖² + λ‖a‖₁`). No usa etiquetas. Guarda `assets/dict_cifar10_32x7.pt`.

In [ ]:
!python -m scripts.pretrain_dictionary --dataset cifar10 --epochs 20 --lr 1e-2 --n_iters 10 --out assets/dict_cifar10_32x7.pt

## 5. Visualizar los átomos — ¿salen Gabor-like?

Control de sanidad de Olshausen-Field: los átomos deberían ser bordes localizados y orientados.

In [ ]:
!python -m scripts.visualize_dictionary --dict assets/dict_cifar10_32x7.pt --out figures/dictionary_cifar10.png
from IPython.display import Image
Image('figures/dictionary_cifar10.png')

## 6. FASE 2 — la curva de datos completa (con etiquetas, Φ congelado)

Condiciones: A (David) · A' (dumb) · A'' (lesión) · Bf (cortical fijo aleatorio) · **Bp (cortical PRETRAINED)** · B (cortical aprendido) · C (Goliat) · D (Goliat+cortical). 3 seeds para barras de error.

El `--dict_path` solo se inyecta en **Bp**.

In [ ]:
!python -m scripts.run_data_curve --dataset cifar10 \
    --dict_path assets/dict_cifar10_32x7.pt \
    --conditions A Ap App Bf Bp B C D \
    --seeds 42 43 44 --epochs 30 \
    --out results/data_curve_cifar10.csv

In [ ]:
# 7. Figura de la curva
!python -m scripts.make_figures --csv results/data_curve_cifar10.csv --out figures/data_curve_cifar10.png
from IPython.display import Image
Image('figures/data_curve_cifar10.png')

In [ ]:
# 8. Descargar resultados al ordenador
from google.colab import files
files.download('results/data_curve_cifar10.csv')
files.download('figures/data_curve_cifar10.png')
files.download('figures/dictionary_cifar10.png')
files.download('assets/dict_cifar10_32x7.pt')